# Lab 01 · Provision your AI Gateway

Create the shared AI Gateway that every later lab calls: an `aigateways` resource with a system-assigned managed
identity, plus the connector gateway that the portal creates alongside it. This notebook is the Python track of the
[lab guide](README.md): it deploys the same [infra/gateway.bicep](infra/gateway.bicep) as the Bicep track, then runs the
same checks.

**Register providers → Create resource group → Deploy Bicep → Read runtime URL → Call it**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| AI Gateway (`Microsoft.ApiManagement/aigateways`) | Runs the data plane at `https://<name>.<region>-01.ai.gateway.azure.com`. Models, tools, keys, and policies hang off it. |
| Connector gateway (`Microsoft.Web/connectorGateways`) | Companion resource with the same name. The portal creates it first, so the template does too. |
| System-assigned managed identity | The gateway's own Microsoft Entra identity. Lab 02 grants it access to Foundry so the gateway calls models without keys. |
| AI Gateway portal ([ai.gateway.azure.com](https://ai.gateway.azure.com)) | Where you manage the gateway after it exists. |
| [infra/gateway.bicep](infra/gateway.bicep) | Declares both resources. The Bicep + azd track deploys it through `infra/main.bicep`; the notebook deploys it on its own. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![AI Gateway architecture: apps and agents call one gateway that governs models and tools](../assets/diagrams/architecture.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- Permission to register the `Microsoft.ApiManagement` and `Microsoft.Web` resource providers, or have them
  registered already.

This is the first notebook in the series. Notebooks 02, 04, 06, and 11 reuse the gateway it creates, so keep it until
you finish them.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q --upgrade azure-identity requests

### 0. Initialize the lab

`utils.Lab` reads your Azure CLI sign-in and sets the region, resource group, and gateway name. The defaults match
every notebook in the series, so later notebooks find this gateway without edits. The gateway name is derived from
your subscription, so it's unique to you. To change any of them, set `AIGW_LOCATION`, `AIGW_RESOURCE_GROUP`, or
`AIGW_GATEWAY_NAME` before you start the kernel. The helpers live in [../shared/utils.py](../shared/utils.py).

In [ ]:
import sys
sys.path.insert(0, "../shared")   # helpers that every lab notebook shares
import utils

lab = utils.Lab("01-provision-gateway", needs_gateway=False)   # this notebook creates the gateway

### 1. Register the resource providers

AI Gateway uses two resource providers: `Microsoft.ApiManagement` for the gateway and `Microsoft.Web` for the
connector gateway. Registration is a one-time step per subscription and takes a minute or two.

In [ ]:
lab.register_providers("Microsoft.ApiManagement", "Microsoft.Web")

### 2. Create the resource group

The template deploys into a resource group. The notebook reuses the group if it exists. It tags the group only when
it creates it, so clean-up can tell lab resources apart.

In [ ]:
lab.create_resource_group()

### 3. Deploy the gateway with Bicep

[infra/gateway.bicep](infra/gateway.bicep) declares the connector gateway and then the AI gateway, with a system-assigned
managed identity and public network access. `lab.deploy` takes the template's parameters as keyword arguments, runs
`az deployment group create`, and prints progress until Azure Resource Manager finishes. Creating the gateway takes
a few minutes. Rerunning the cell deploys the same template again, which leaves an existing gateway as it is.

> You can set the template's `backendSubnetId` parameter for outbound virtual network integration only when you
> create the gateway. Lab 18 covers private networking.

In [ ]:
outputs = lab.deploy("infra/gateway.bicep", gatewayName=lab.gateway_name, location=lab.location, tags=lab.tags)

### 4. Read the runtime URL and managed identity

The template's outputs carry the runtime URL, which it reads from `properties.frontend.defaultHostname`. Always read
the host name rather than building it. Apps call models under `/default/models/openai/v1/` and tools under
`/default/toolservers/<name>/mcp`, where `default` is the workspace. Note the managed identity's principal ID:
lab 02 grants it the Foundry User role.

In [ ]:
import re

gateway_url  = outputs["gatewayUrl"]
principal_id = outputs["principalId"]

print(f"🌐 Runtime URL:      {gateway_url}")
print(f"🤖 Models endpoint:  {gateway_url}/default/models/openai/v1/")
print(f"🧰 Tools endpoint:   {gateway_url}/default/toolservers/<name>/mcp")
print(f"🪪 Principal ID:     {principal_id}")

host = gateway_url.removeprefix("https://").lower()
assert re.fullmatch(rf"{re.escape(lab.gateway_name.lower())}\.[a-z0-9]+-\d+\.ai\.gateway\.azure\.com", host), host
assert principal_id, "The gateway should have a system-assigned managed identity."

### 5. Call the runtime URL

Send a request without a key. Any HTTP status below 500, such as `400`, `401`, or `404`, proves that DNS, TLS, and the
gateway are up. You add a model and a key in the next notebook. A brand-new host name can take a minute to resolve,
so `utils.wait_for_endpoint` retries connection errors.

In [ ]:
url = f"{gateway_url}/default/models/openai/v1/chat/completions"
response = utils.wait_for_endpoint(url)

print(f"📨 GET {url} -> HTTP {response.status_code}")
assert response.status_code < 500, response.text[:500]
print("✅ The runtime endpoint answers over HTTPS")

### 6. Explore the portal

Open the [AI Gateway portal](https://ai.gateway.azure.com) and select your gateway in the switcher in the top bar.

- **Home** shows the setup guide for the new gateway.
- **Managed identities** › **Configure identities** shows the system-assigned identity. Its principal ID matches the
  value printed in step 4.
- **Models**, **Tools**, **Policies**, and **Keys** are empty for now. Notebooks 02, 04, and 06 fill them in.

The [lab guide](README.md#tour-the-portal) has a table of every portal area and the lab that covers it.

### 7. Clean up resources

Keep the gateway: notebooks 02, 04, 06, and 11 reuse it. When you finish the series, uncomment the lines below to
delete the gateway, then the connector gateway. Gateway deletion takes about 10 minutes. Delete the resource group
only if this notebook created it and nothing else lives in it.

In [ ]:
# Later notebooks reuse the gateway. Uncomment to delete it when you're done with the series.
# lab.delete(lab.gateway_id, wait=True)
# lab.delete(lab.connector_gateway_id, api_version=utils.CONNECTOR_API)
if lab.arm("GET", lab.gateway_id):
    print(f"ℹ️ Kept gateway {lab.gateway_name} and its connector gateway for the next notebooks")

## Troubleshooting

- **`The subscription is not registered to use namespace`**: rerun step 1, or run
  `az provider register -n Microsoft.ApiManagement` and `az provider register -n Microsoft.Web`.
- **`LocationNotAvailableForResourceType`**: set `AIGW_LOCATION` to a region from [Find the available regions](../SETUP.md#find-the-available-regions).
- **`AuthorizationFailed`**: you need Contributor on the resource group. Check with
  `az role assignment list --assignee <your-upn> -g <resource-group> -o table`.
- **Name rejected or already taken**: the name becomes the first label of the host name. Use up to 24 letters, digits,
  and hyphens, and start with a letter. Set `AIGW_GATEWAY_NAME` to try another name.
- **`Deployment ... failed`**: the cell prints each failed resource with its error. The resource group's
  **Deployments** page in the Azure portal has the full details.
- **`Bicep CLI not found`**: run `az bicep install`, then rerun step 3.
- **`ConnectionError` or `Could not resolve host`** in step 5: the host name is new. Wait a minute and rerun the cell.

## Next steps

- Continue with [Publish a Foundry model and make your first call](../02-first-model-call/README.md) and its notebook [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb).
- Learn the concepts in [How AI Gateway works](../00-how-ai-gateway-works/README.md).
- Docs: [AI Gateway CLI quickstart](https://ai.gateway.azure.com/docs/quickstarts/cli).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/01-provision-gateway/](https://azure.github.io/ai-gateway/labs/01-provision-gateway/).